# 01 - Busqueda MGCECDL Clasificacion con Optuna

Este notebook clona el repositorio si hace falta, procesa los datos desde `data` y genera el objeto `Study` de Optuna para MGCECDL clasificacion.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/jclugor/chec-local-uiti-vano-interpreter.git"
REPO_NAME = "chec-local-uiti-vano-interpreter"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


def ensure_lfs_data(project_root):
    data_path = project_root / "data" / "Indicadores_vano_v3.csv"
    if shutil.which("git-lfs") and (project_root / ".git").exists():
        subprocess.run(["git", "lfs", "install", "--local"], cwd=project_root, check=False)
        subprocess.run(["git", "lfs", "pull"], cwd=project_root, check=True)
    if data_path.exists() and data_path.stat().st_size < 1024:
        head = data_path.read_text(errors="ignore")[:120]
        if "git-lfs" in head:
            raise RuntimeError(
                "Indicadores_vano_v3.csv quedo como puntero Git LFS. "
                "Descarga el archivo LFS antes de continuar."
            )


PROJECT_ROOT = resolve_project_root()
install_project_requirements(PROJECT_ROOT)
ensure_lfs_data(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OPTUNA_DIR = DATA_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CUDA disponible:", torch.cuda.is_available())
print("MPS disponible:", torch.backends.mps.is_available())

In [ ]:
from chec_impacto.data import (
    construir_matriz_adyacencia_mgcecdl,
    preparar_splits_estratificados,
    procesar_dataset_completo,
)
from chec_impacto.training import construir_modalidades_mgcecdl, resolve_training_device

VENTANA_CLIMATICA_HORAS = 12
FILTRO_UITI_MAX = None

DATA_PATH_CANDIDATES = [
    DATA_DIR / "Indicadores_vano_v3.csv",
    DATA_DIR / "Indicadores_vano_v2.csv",
    DATA_DIR / "Indicadores_vano_v1.csv",
]
DATASET_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("No se encontro Indicadores_vano_v3/v2/v1.csv en data/.")

VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
DEVICE = resolve_training_device("auto")
print(f"Usando device: {DEVICE}")

datos_procesados = procesar_dataset_completo(
    path_clima=DATASET_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target="UITI_VANO",
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

X = datos_procesados["X"]
y = datos_procesados["y"]
features = datos_procesados["features"]

modality_feature_indices = construir_modalidades_mgcecdl(features)
print(
    "Modos MGCECDL para busqueda/entrenamiento:",
    {name: len(indices) for name, indices in modality_feature_indices.items()},
)

GRAPH_DIR = DATA_DIR / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
graph_files = {
    "matrix": GRAPH_DIR / "mgcecdl_adjacency_matrix.npy",
    "features": GRAPH_DIR / "mgcecdl_feature_order.json",
    "edges": GRAPH_DIR / "mgcecdl_preserved_edges.json",
}
if all(path.exists() for path in graph_files.values()):
    stored_features = json.loads(graph_files["features"].read_text(encoding="utf-8"))
    if list(stored_features) == list(features):
        graph_adjacency_matrix = np.load(graph_files["matrix"])
        graph_preserved_edges = json.loads(graph_files["edges"].read_text(encoding="utf-8"))
        print(f"Grafo MGCECDL cargado desde: {GRAPH_DIR}")
    else:
        print("Grafo existente incompatible con las features actuales. Se reconstruirá en data/graphs.")
        graph_adjacency_matrix, graph_preserved_edges = construir_matriz_adyacencia_mgcecdl(
            features,
            ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
        )
else:
    graph_adjacency_matrix, graph_preserved_edges = construir_matriz_adyacencia_mgcecdl(
        features,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )

if not all(path.exists() for path in graph_files.values()) or list(json.loads(graph_files["features"].read_text(encoding="utf-8"))) != list(features):
    np.save(graph_files["matrix"], graph_adjacency_matrix)
    graph_files["features"].write_text(
        json.dumps(features, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    graph_files["edges"].write_text(
        json.dumps(graph_preserved_edges, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Grafo MGCECDL guardado/actualizado en: {GRAPH_DIR}")
graph_adjacency_tensor = torch.as_tensor(
    graph_adjacency_matrix,
    dtype=torch.float32,
    device=DEVICE,
)
assert graph_adjacency_matrix.shape == (len(features), len(features))
print("Matriz de adyacencia MGCECDL:", graph_adjacency_matrix.shape)
print("Aristas preservadas:", len(graph_preserved_edges))


In [ ]:
from chec_impacto.training import (
    buscar_estudio_optuna_mgcecdl,
    calcular_estadisticas_reconstruccion_mgcecdl,
    escalar_features_minmax_mgcecdl,
    guardar_estudio_optuna,
)

MODO_OBJETIVO = "clasificacion"
N_TRIALS = 40
MAX_EPOCHS = 60
PATIENCE = 40
RANDOM_STATE = 42
WEIGHT_MODALITY_LOSS_BY_RELIABILITY = True

splits = preparar_splits_estratificados(
    X,
    y,
    modo=MODO_OBJETIVO,
    random_state=RANDOM_STATE,
)
splits = escalar_features_minmax_mgcecdl(splits)
print(
    "Rango X_train MGCECDL escalado:",
    float(np.min(splits["X_train"])),
    float(np.max(splits["X_train"])),
)
n_classes = int(len(np.unique(splits["y_categories_original"])))
feature_mean, feature_std = calcular_estadisticas_reconstruccion_mgcecdl(
    splits["X_train"]
)

study = buscar_estudio_optuna_mgcecdl(
    modo_objetivo=MODO_OBJETIVO,
    storage_path=OPTUNA_DIR / "mgcecdl_classification_feature_attention_params.journal",
    X_train=splits["X_train"],
    y_train=splits["y_train"],
    X_valid=splits["X_valid"],
    y_valid=splits["y_valid"],
    modality_feature_indices=modality_feature_indices,
    feature_mean=feature_mean,
    feature_std=feature_std,
    adjacency_matrix=graph_adjacency_matrix,
    n_classes=n_classes,
    device=DEVICE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=RANDOM_STATE,
    weight_modality_loss_by_reliability=WEIGHT_MODALITY_LOSS_BY_RELIABILITY,
    n_trials=N_TRIALS,
)
guardar_estudio_optuna(
    study,
    OPTUNA_DIR / "mgcecdl_optuna_study_clasificacion_feature_attention.pkl",
)
study